In [45]:
import pandas as pd
import numpy as np
import pickle

In [16]:
fname  = "WO2020245233A1"
tables = pickle.load(open(f'data/processed_tables/{fname}.pkl', 'rb'))

In [17]:
dfs = []
for table in tables:
    df = pd.DataFrame(table['table'])
    dfs.append(df)


In [20]:
def is_relevant_table(df):
    cols = df.columns
    if any(["Oligonucleotide" in col for col in cols]):
        return True
    return False


relevant_tables = []
for i, df in enumerate(dfs):
    if filter_relevant_tables(df):
        print(i, "\n#################")
        print(df.head())
        relevant_tables.append(df)    

1 
#################
  SEQ ID NO  motif sequence start  end  design CMP ID NO  \
0         4  aagaaaccaaaccc   743  756  2-10-2       4_1   
1         5  aaagaaaccaaacc   744  757  2-10-2       5_1   
2         6  aaaagaaaccaaac   745  758  2-10-2        61   
3         7  caaaagaaaccaaa   746  759  2-10-2       7_1   
4         8  ccaaaagaaaccaa   747  760  2-10-2        81   

  Oligonucleotide compound  
0           AAgaaaccaaacCC  
1           AAagaaaccaaaCC  
2           AAaagaaaccaaAC  
3           CAaaagaaaccaAA  
4           CCaaaagaaaccAA  
3 
#################
  SEQID   CMPID Oligonucleotide Base Sequence Oligonucleotide compound  \
0  1099  1099_1            CCAAAAGAAACCAAACCC       CCAAaagaaaccaaacCC   
1  1100  1100_1           CCCCATTCAAATATTTATT      CCccattcaaatatttATT   
2  1101  1101_1             AATCATTTACCCCCAAC        AAtcatttaccccCAAC   
3  1102  1102_1            TATCTCAAACTATCCCCA       TAtctcaaactatcccCA   
4  1103  1103_1                                     T

In [59]:
# get in vitro KD data
df = pd.concat(relevant_tables[1:], axis=0)
print("Any duplicates?", df.duplicated().any())


# Clean and prepare the target variable
# Convert target to numeric, coercing errors to NaN
df['% of ATXN3 mRNA remaining'] = pd.to_numeric(df['% of ATXN3 mRNA remaining'], errors='coerce')

# Drop rows with missing target values
df.dropna(subset=['% of ATXN3 mRNA remaining'], inplace=True)

# Ensure all sequences are uppercase for consistency in featurization
df['Oligonucleotide Base Sequence'] = df['Oligonucleotide Base Sequence'].str.upper()


Any duplicates? False


In [58]:
X = df['Oligonucleotide Base Sequence']
y = df['% of ATXN3 mRNA remaining']
y = y.div(100)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTraining set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")


Training set size: 252
Test set size: 63
